In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

# load data

In [2]:
# Load the data
train = pd.read_csv('train.data.csv')
test  = pd.read_csv('test.data.csv')
# Define features and target
features = ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot']
target   = 'price'

In [3]:
# feature extraction
X_train = train[features]
y_train = train[target]
X_test  = test[features]
y_test  = test[target]

# (a)

In [4]:
# fit the linear regression model
lr = LinearRegression()
lr.fit(X_train, y_train)

# print results
print("=== Part (a) sklearn OLS ===")
print(" R-square train:", r2_score(y_train, lr.predict(X_train)))
print(" R-square test: ", r2_score(y_test,  lr.predict(X_test)))

=== Part (a) sklearn OLS ===
 R-square train: 0.5101138530794578
 R-square test:  0.5049944614037095


# (b)

In [5]:
# load the fancy house data
fancy = pd.read_csv('fancyhouse.csv')

In [6]:
# predict the price of the fancy house
pred_fancy = lr.predict(fancy[features])[0]
print("=== Part (b) fancy house prediction ===")
print("\nBill Gates' house features:")
print(fancy[features].to_string(index=False))
print(" Predicted price:", pred_fancy)

=== Part (b) fancy house prediction ===

Bill Gates' house features:
 bedrooms  bathrooms  sqft_living  sqft_lot
        8         25        50000    225000
 Predicted price: 15436769.53822418


# (c)

In [7]:
# define interaction feature
for df in (train, test, fancy):
    df['bed_bath'] = df['bedrooms'] * df['bathrooms']

In [8]:
# add interaction feature to the model
features2 = features + ['bed_bath']
X2_train = train[features2]
X2_test  = test[features2]

In [9]:
# fit linear regression with interaction feature
lr2 = LinearRegression()
lr2.fit(X2_train, y_train)
print("=== Part (c) with interaction feature ===")
print(" R-square train:", r2_score(y_train, lr2.predict(X2_train)))
print(" R-square test: ", r2_score(y_test,  lr2.predict(X2_test)))
print(" Fancy house price:", lr2.predict(fancy[features2])[0])

=== Part (c) with interaction feature ===
 R-square train: 0.5173532927738305
 R-square test:  0.5105355458590048
 Fancy house price: 18607312.890443005


# (d)

In [10]:
def ols_estimate(X, y, fit_intercept=True):
    """
    Compute OLS coefficients via the normal equations:

    Parameters
    ----------
    X : array_like, shape (n_samples, n_features)
        Predictor matrix (no intercept column).
    y : array_like, shape (n_samples,)
        Response vector.
    fit_intercept : bool, default=True
        If True, prepend a column of ones to X for the intercept term.

    Returns
    -------
    beta_hat : ndarray, shape (n_features + int(fit_intercept),)
        Estimated coefficients.  If fit_intercept=True, beta_hat[0] is the intercept.
    """
    X_mat = np.asarray(X)
    if fit_intercept:
        # add a column of ones at the front for the intercept
        ones = np.ones((X_mat.shape[0], 1))
        X_mat = np.hstack([ones, X_mat])

    # compute normal‐equations components
    XtX = X_mat.T @ X_mat      # (p×n) × (n×p) → p×p
    Xty = X_mat.T @ y          # (p×n) × (n,)   → p

    # solve for β̂
    beta_hat = np.linalg.solve(XtX, Xty)
    return beta_hat
def gd_regression(X_mat, y_vec, lr=1e-4, epochs=20000, tol=1e-3):
    """
    Fit y ≈ Xbeta by gradient descent.
    
    X_mat : (n_samples, n_features) design matrix (already standardized)
    y_vec : (n_samples,) target vector (already centered & scaled)
    lr    : learning rate
    epochs: maximum GD steps
    tol   : stop when gradient norm < tol
    """


    n_feats = X_mat.shape[1]
    # initialize beta to zero
    beta = np.zeros(n_feats)
    # gradient descent loop
    for i in range(epochs):
        # compute the residuals
        resid = X_mat.dot(beta) - y_vec
        # compute the gradient
        grad = X_mat.T.dot(resid)  
        # compute the norm of the gradient
        gnorm = np.linalg.norm(grad)
        # check for convergence
        if gnorm < tol:
            break
        # update beta using the gradient and learning rate
        beta -= lr * grad
    return beta

def standardize(df):
    """Return standardized copy plus means and stds."""
    means = df.mean(axis=0)
    stds  = df.std(axis=0, ddof=0).replace(0,1)
    return (df - means) / stds, means, stds


In [11]:
#Prepare data for model without interaction
raw_feats = ['bedrooms','bathrooms','sqft_living','sqft_lot']
Xtr_raw = train[raw_feats]
ytr_raw = train['price']
Xte_raw = test[raw_feats]
yte_raw = test['price']

In [12]:
# Standardize
Xtr, mu_X, sigma_X = standardize(Xtr_raw)
Xte = (Xte_raw - mu_X) / sigma_X
ytr, mu_y, sigma_y = standardize(ytr_raw.to_frame())
ytr = ytr.values.ravel()
yte = (yte_raw - mu_y['price']) / sigma_y['price']

In [13]:
# Fit GD
beta_gd = gd_regression(Xtr.values, ytr, lr=5e-5, epochs=10000, tol=1e-2)

In [14]:
# Predictions (unscale output)
ytr_pred_std = Xtr.values.dot(beta_gd)
yte_pred_std = Xte.values.dot(beta_gd)
ytr_pred = ytr_pred_std*sigma_y['price'] + mu_y['price']
yte_pred = yte_pred_std*sigma_y['price'] + mu_y['price']

print("GD Model (no interaction)")
print(" Train R²:", r2_score(ytr_raw, ytr_pred))
print(" Test R²:", r2_score(yte_raw, yte_pred))
print(' gd estimate witout interation term:', beta_gd)


# --- Predict Bill Gates' house ---
fancy_feats = fancy[raw_feats]
fancy_scaled = (fancy_feats - mu_X) / sigma_X
fancy_pred_std = fancy_scaled.values.dot(beta_gd)
fancy_price = fancy_pred_std*sigma_y['price'] + mu_y['price']
print("Predicted Bill Gates' house price:", fancy_price[0])

GD Model (no interaction)
 Train R²: 0.5101138530779468
 Test R²: 0.5049944961439965
 gd estimate witout interation term: [-0.15146983  0.00773792  0.79183088 -0.04514334]
Predicted Bill Gates' house price: 15436751.700040579


In [15]:
# --- Now add interaction feature ---
train_int = train.copy()
train_int['bed_x_bath'] = train_int['bedrooms'] * train_int['bathrooms']
test_int  = test.copy()
test_int ['bed_x_bath'] = test_int ['bedrooms'] * test_int ['bathrooms']

# Prepare data for model with interaction
feat_int = raw_feats + ['bed_x_bath']
Xtr2_raw = train_int[feat_int]
ytr2_raw = train_int['price']
Xte2_raw = test_int[ feat_int]
yte2_raw = test_int[ 'price']
 # Standardize
Xtr2, mu_X2, sigma_X2 = standardize(Xtr2_raw)
Xte2 = (Xte2_raw - mu_X2) / sigma_X2
ytr2, mu_y2, sigma_y2 = standardize(ytr2_raw.to_frame())
ytr2 = ytr2.values.ravel()
yte2 = (yte2_raw - mu_y2['price']) / sigma_y2['price']

# Fit GD with interaction term
beta_gd2 = gd_regression(Xtr2.values, ytr2, lr=9e-6, epochs=20000, tol=1e-3)

ytr2_pred = Xtr2.values.dot(beta_gd2)*sigma_y2['price'] + mu_y2['price']
yte2_pred = Xte2.values.dot(beta_gd2)*sigma_y2['price'] + mu_y2['price']

print("GD Model (plus interaction)")
print(" Train R²:", r2_score(ytr2_raw, ytr2_pred))
print(" Test R²:", r2_score(yte2_raw, yte2_pred))
print(' gd estimate with interation term:', beta_gd2)

GD Model (plus interaction)
 Train R²: 0.5173532927736819
 Test R²: 0.5105355505069821
 gd estimate with interation term: [-0.32877123 -0.23326249  0.77192761 -0.04491516  0.38964673]


In [16]:
print('ols_estimate witout interation term: ', ols_estimate(Xtr, ytr, fit_intercept=True)[1:])
print('ols_estimate witth interation term: ', ols_estimate(Xtr2, ytr2, fit_intercept=True)[1:])

ols_estimate witout interation term:  [-0.15147017  0.00773629  0.7918328  -0.04514359]
ols_estimate witth interation term:  [-0.32877205 -0.23326364  0.77192759 -0.04491517  0.38964848]


# (e)

In [17]:
# def train_sgd_lr(X, y, learning_rate=1e-3, epochs=10000):
#     """
#     Fit a linear model via stochastic gradient descent (no intercept term).
#     Parameters
#     ----------
#     X : array_like, shape (n_samples, n_features)
#         Feature matrix (already scaled if desired).
#     y : array_like, shape (n_samples,)
#         Target values.
#     learning_rate : float
#         Step size for updates.
#     epochs : int
#         Total number of SGD steps.
#     Returns
#     -------
#     weights : ndarray, shape (n_features,)
#         Learned coefficient vector.
#     """
#     n_samples, n_features = X.shape
#     weights = np.zeros(n_features)
    
#     # Loop over epochs
#     for _ in range(epochs):
#         # Randomly select a sample
#         i = np.random.randint(n_samples)
#         # Compute the gradient and update weights
#         xi = X[i]
#         # Compute the prediction error
#         error = xi.dot(weights) - y[i]
#         # Update the weights using the gradient
#         grad = xi * error
#         # Update the weights using the learning rate
#         weights -= learning_rate * grad
        
#     return weights


# ——— SGD solver ———
def sgd_regressor(X, y, learning_rate=5e-4, steps=20000):
    """
    Runs plain SGD on MSE loss, including intercept.
    Returns weight vector of length D+1.
    """
    n, d = X.shape
    # add intercept column
    X_aug = np.hstack([np.ones((n,1)), X])
    weights = np.zeros(d+1)

    for _ in range(steps):
        i = np.random.randint(n)
        xi = X_aug[i]
        yi = y[i]
        error = xi.dot(weights) - yi
        grad = 2 * error * xi
        weights -= learning_rate * grad

    return weights

In [18]:
# ——— Load data ———
df_train = pd.read_csv('train.data.csv')
df_test  = pd.read_csv('test.data.csv')
df_fancy = pd.read_csv('fancyhouse.csv')

# ——— Feature construction ———
base_features = ['bedrooms','bathrooms','sqft_living','sqft_lot']
df_train['bed_bath'] = df_train['bedrooms'] * df_train['bathrooms']
df_test['bed_bath']  = df_test['bedrooms']  * df_test['bathrooms']
df_fancy['bed_bath'] = df_fancy['bedrooms'] * df_fancy['bathrooms']

extended_features = base_features + ['bed_bath']

# ——— Extract arrays ———
X_tr_base = df_train[base_features].values;  y_tr = df_train['price'].values
X_te_base = df_test[base_features].values;   y_te = df_test['price'].values
X_fh_base = df_fancy[base_features].values

X_tr_ext = df_train[extended_features].values
X_te_ext = df_test[extended_features].values
X_fh_ext = df_fancy[extended_features].values

# ——— Scaling ———
scaler_base = StandardScaler().fit(X_tr_base)
X_tr_b = scaler_base.transform(X_tr_base)
X_te_b = scaler_base.transform(X_te_base)
X_fh_b = scaler_base.transform(X_fh_base)

scaler_ext = StandardScaler().fit(X_tr_ext)
X_tr_e = scaler_ext.transform(X_tr_ext)
X_te_e = scaler_ext.transform(X_te_ext)
X_fh_e = scaler_ext.transform(X_fh_ext)

In [31]:
# ——— Train & evaluate on base features ———
w_base = sgd_regressor(X_tr_b, y_tr, learning_rate=8e-4, steps=18000)
X_tr_b_aug = np.hstack([np.ones((X_tr_b.shape[0],1)), X_tr_b])
X_te_b_aug = np.hstack([np.ones((X_te_b.shape[0],1)), X_te_b])
X_fh_b_aug = np.hstack([np.ones((X_fh_b.shape[0],1)), X_fh_b])

y_tr_pred_b = X_tr_b_aug.dot(w_base)
y_te_pred_b = X_te_b_aug.dot(w_base)

print("SGD R-square train:", r2_score(y_tr, y_tr_pred_b))
print("SGD R-square test: ", r2_score(y_te, y_te_pred_b))

# ——— Train & evaluate on extended features ———
w_ext = sgd_regressor(X_tr_e, y_tr, learning_rate=9e-4, steps=15000)
X_tr_e_aug = np.hstack([np.ones((X_tr_e.shape[0],1)), X_tr_e])
X_te_e_aug = np.hstack([np.ones((X_te_e.shape[0],1)), X_te_e])
X_fh_e_aug = np.hstack([np.ones((X_fh_e.shape[0],1)), X_fh_e])

y_tr_pred_e = X_tr_e_aug.dot(w_ext)
y_te_pred_e = X_te_e_aug.dot(w_ext)


print("SGD (interaction) R-sqaure train:", r2_score(y_tr, y_tr_pred_e))
print("SGD (interaction) R-square test: ", r2_score(y_te, y_te_pred_e))

SGD R-square train: 0.5091199465307769
SGD R-square test:  0.5029430835056805
SGD (interaction) R-sqaure train: 0.5146868146757181
SGD (interaction) R-square test:  0.5081965642056857
